<a href="https://colab.research.google.com/github/ceuratfmg2mai/fakereviews/blob/colab/05.%20evaluacion_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Categorización de reseñas con el uso de una base de datos vectorial aplicando la técnica de similitud

En esta etapa del TFM, emplearemos una base de datos vectorial con el objetivo de aplicar técnicas de búsqueda por similitud. Esto nos permitirá analizar la relevancia de las reseñas recuperadas y examinar la coherencia entre su similitud textual y su clasificación preexistente ('Fake' o 'Genuine').

Para ello, utilizaremos la base de datos vectorial Qdrant con el fin de encontrar reseñas de Yelp que sean textualmente parecidas entre sí, según la interpretación de su significado realizada por un modelo de embeddings. Cabe destacar que estas reseñas ya fueron clasificadas previamente por humanos como 'Fake' o 'Genuine' y esta clasificación será almacenada en Qdrant.

Al introducir nuevas reseñas como consulta en Qdrant, compararemos los hallazgos de la plataforma (*es decir, las reseñas que Qdrant identifica como semánticamente similares, mostrando la clasificación ('Fake' o 'Genuine')*) con las evaluaciones realizadas por humanos. Estas evaluaciones humanas definirán tanto la relevancia de una reseña para la consulta específica como su verdadera clasificación ('Fake' o 'Genuine').

Este cruce permitirá no solo medir la efectividad de Qdrant para identificar contenido textualmente relevante, sino también evaluar la precisión de las clasificaciones originales de DeepSeek sobre dicho contenido semánticamente similar. Así, se podrán revelar patrones y posibles áreas de mejora tanto en el sistema de búsqueda por similitud como en el proceso de clasificación de reseñas.

# Instalación

In [6]:
!pip install -U langchain-community  > /dev/log 2>&1
!pip install -U qdrant-client > /dev/log 2>&1

# Librerías

In [28]:
import polars as pl
import pandas as pd
import pyarrow
import matplotlib.pyplot as plt
plt.style.use('Solarize_Light2')
import seaborn as sns
import time
start_time_global = time.time()
import os
import sys
from google.colab import userdata
from google.colab import drive
from importlib import metadata
import uuid

from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient, models

color = '\033[1m\033[38;5;208m'
print(f"{color}Versión pandas: {pd.__version__}")
print(f"{color}Versión polars: {pl.__version__}")
print(f"{color}Versión pyarrow: {pyarrow.__version__}")
print(f"{color}Versión langchain: {metadata.version('langchain')}")
print(f"{color}Versión qdrant_client: {metadata.version('qdrant_client')}")

Versión pandas: 2.2.2
Versión polars: 1.21.0
Versión pyarrow: 18.1.0
Versión langchain: 0.3.25
Versión qdrant_client: 1.14.2


# Inicialización de las variables de entorno

In [29]:
# 1. Configurar Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # Generates vector embeddings for each chunk
# Proba el modelo y pedir que genere el vector para la palabra test
# Número de elementos que contiene esa lista.
vector_size_test = len(embedding_model.embed_query("test"))
print(vector_size_test)

384


In [15]:
drive.mount('/content/drive', force_remount=True)
explicit_work_path = '/content/drive/MyDrive/Colab Notebooks/tfm_grupo_2/'
print(os.listdir(explicit_work_path))

Mounted at /content/drive
['base_reviews.json', 'yelp_academic_dataset_review_selected_0_255.jsonl', 'final_reviews_categorize_0_255.jsonl', 'data_yelp', 'yelp_academic_dataset_review_selected.jsonl', 'yelp_academic_dataset_review_selected.arrow', 'yelp_academic_dataset_review_selected.csv', 'yelp_academic_dataset_review_selected_prompt.arrow', 'yelp_academic_dataset_review_selected_prompt.jsonl', 'final_reviews_categorize.jsonl', 'yelp_academic_dataset_review_final_selected_1900.jsonl', 'yelp_academic_dataset_review_golden_evaluador_2.json', 'yelp_academic_dataset_review_golden_evaluador_1.jsonl', 'ground_truth.jsonl']


In [23]:
try:
    qdrant_key = userdata.get("QDRANT_KEY")
    qdrant_url = 'https://73e55abe-f0f4-4f08-b30d-c1e2187ebc45.europe-west3-0.gcp.cloud.qdrant.io:6333'
    print("Datos de Qdrant cargados.")
except KeyError:
    print("Error: Las variables de entorno.")

Datos de Qdrant cargados.


In [25]:
review_categorice_file_path = f'{explicit_work_path}yelp_academic_dataset_review_final_selected_1900.jsonl'
df_data_reviews_categorice_pl = pl.read_json(review_categorice_file_path)
print(f"\nTotal de reseñas categorizadas: {df_data_reviews_categorice_pl.shape[0]}")


Total de reseñas categorizadas: 2758


In [26]:
# Nombre de la conlección
COLLECTION_NAME = "tfm2g"

In [27]:
# --- Inicialización del cliente de Qdrant ---
qdrant_cloud_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_key,
    timeout=30
)